In [1]:
import os
import json
import pandas as pd


pd.set_option("display.max_columns", None)
# pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [2]:
base_path = "../results"

folders = {

    3: "validation_eval_service_3",
    5: "validation_eval_service_5",
    8: "validation_eval_service_8",
    10: "validation_eval_service_10"

}

In [3]:
data = {}

for k, folder_name in folders.items():

    folder_path = os.path.join(base_path, folder_name)
#     print(folder_path)

    data[k] = []

    for file_name in os.listdir(folder_path):

        if not file_name.endswith(".json"):
            continue

        if "_efficiency" in file_name:
            continue

        file_path = os.path.join(folder_path, file_name)
#         print(file_path)

        with open(file_path, "r") as f:
            rows = json.load(f)
#             print(rows)

        for row in rows:
            row = row.copy()
            row["source_file"] = file_name
            data[k].append(row)

    print(f"K={k}: loaded {len(data[k])} rows")

K=3: loaded 273 rows
K=5: loaded 339 rows
K=8: loaded 504 rows
K=10: loaded 603 rows


In [4]:
df_k3 = pd.DataFrame(data[3])
df_k5 = pd.DataFrame(data[5])
df_k8 = pd.DataFrame(data[8])
df_k10 = pd.DataFrame(data[10])

print("K=3:", df_k3.shape)
print("K=5:", df_k5.shape)
print("K=8:", df_k8.shape)
print("K=10:", df_k10.shape)

def filter_model(df, model_name):
    return df[df["model_name"] == model_name].copy()

K=3: (273, 28)
K=5: (339, 28)
K=8: (504, 28)
K=10: (603, 28)


In [5]:
# MODEL = "Qwen/Qwen3-Reranker-4B"

MODEL = "mixedbread-ai/mxbai-rerank-large-v2"
# MODEL = "BAAI/bge-reranker-v2-m3"

df_k3 = filter_model(df_k3, MODEL)
df_k5 = filter_model(df_k5, MODEL)
df_k8 = filter_model(df_k8, MODEL)
df_k10 = filter_model(df_k10, MODEL)

In [6]:
# print(len(df_k3))
# print(len(df_k5))
# print(len(df_k8))
# print(len(df_k10))

# Distribution Analysis Before threshold condition

#### to see the pattern how the model predict specialists and general


In [7]:

def distribution_summary(df):

    general = df[
        df["expected_service"] == "general"
    ]["zscore_margin"]

    specialist = df[
        df["expected_service"] != "general"
    ]["zscore_margin"]

    return pd.DataFrame(
        [
            general.describe(),
            specialist.describe()
        ],
        index=[
            "general",
            "specialist"
        ]
    )

In [8]:
distribution_summary(df_k3)

,count,mean,std,min,25%,50%,75%,max
general,50.0,1.531984,0.556722,0.0000,1.453775,1.7513,1.894675,2.1213
specialist,41.0,2.091022,0.075227,1.7059,2.096700,2.1159,2.118900,2.1211


In [9]:
distribution_summary(df_k5)

,count,mean,std,min,25%,50%,75%,max
general,50.0,1.085236,0.752693,0.0000,0.461675,1.08345,1.694375,2.4782
specialist,63.0,2.360029,0.287721,0.6572,2.391400,2.47570,2.487500,2.4980


In [10]:
distribution_summary(df_k8)

,count,mean,std,min,25%,50%,75%,max
general,50.0,1.249404,0.790597,0.0,0.7227,1.21255,1.781000,2.8591
specialist,118.0,2.523035,0.775758,0.0,2.4275,2.89910,2.988825,3.0196


In [11]:
distribution_summary(df_k10)

,count,mean,std,min,25%,50%,75%,max
general,50.0,1.260684,0.909578,0.0000,0.51515,1.15675,1.741525,3.1850
specialist,151.0,2.565295,0.991153,0.0287,2.22550,3.01900,3.268600,3.3303


In [12]:
summary = []

for k, df in {
    3: df_k3,
    5: df_k5,
    8: df_k8,
    10: df_k10,
}.items():

    general = df[
        df["expected_service"] == "general"
    ]["zscore_margin"]

    specialist = df[
        df["expected_service"] != "general"
    ]["zscore_margin"]

    summary.append({
        "K": k,
        "General Median": round(general.median(), 3),
#         "General Std": round(general.std(), 3),
        "Specialist Median": round(specialist.median(), 3),
#         "Specialist Std": round(specialist.std(), 3),
    })

summary_df = pd.DataFrame(summary)

summary_df

,K,General Median,Specialist Median
0,3,1.751,2.116
1,5,1.083,2.476
2,8,1.213,2.899
3,10,1.157,3.019


In [13]:
summary = []

for k, df in {
    3: df_k3,
    5: df_k5,
    8: df_k8,
    10: df_k10,
}.items():

    general = df[
        df["expected_service"] == "general"
    ]

    specialist = df[
        df["expected_service"] != "general"
    ]

    summary.append({
        "K": k,

        "General Margin": round(
            general["margin"].median(), 3
        ),
        "Specialist Margin": round(
            specialist["margin"].median(), 3
        ),

        "General Top Z": round(
            general["top_zscore"].median(), 3
        ),
        "Specialist Top Z": round(
            specialist["top_zscore"].median(), 3
        ),

        "General Z-Margin": round(
            general["zscore_margin"].median(), 3
        ),
        "Specialist Z-Margin": round(
            specialist["zscore_margin"].median(), 3
        ),
    })

pd.DataFrame(summary)

,K,General Margin,Specialist Margin,General Top Z,Specialist Top Z,General Z-Margin,Specialist Z-Margin
0,3,0.005,0.962,1.370,1.414,1.751,2.116
1,5,0.003,0.921,1.576,2.000,1.083,2.476
2,8,0.005,0.794,2.123,2.641,1.213,2.899
3,10,0.005,0.741,2.235,2.978,1.157,3.019


## Threshold cutting based on quartiles

In [14]:
summary = []

for k, df in {
    3: df_k3,
    5: df_k5,
    8: df_k8,
    10: df_k10,
}.items():

    general = df[
        df["expected_service"] == "general"
    ]["zscore_margin"]

    specialist = df[
        df["expected_service"] != "general"
    ]["zscore_margin"]

    g_q25 = general.quantile(0.25)
#     print("g_q25",g_q25)
    g_q75 = general.quantile(0.75)
#     print("g_q75",g_q75)
    
    
    s_q25 = specialist.quantile(0.25)
    s_q75 = specialist.quantile(0.75)


    summary.append({
        "K": k,

        "General Q25": round(g_q25, 3),
        "General Q75": round(g_q75, 3),

        "Specialist Q25": round(s_q25, 3),
        "Specialist Q75": round(s_q75, 3),
        
        "Gap": round(s_q25 - g_q75, 3)

    })

pd.DataFrame(summary)

,K,General Q25,General Q75,Specialist Q25,Specialist Q75,Gap
0,3,1.454,1.895,2.097,2.119,0.202
1,5,0.462,1.694,2.391,2.488,0.697
2,8,0.723,1.781,2.428,2.989,0.647
3,10,0.515,1.742,2.226,3.269,0.484
